In [1]:
import sys
import tensorflow as tf

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

2026-10-08 19:06:23.523672: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-08 19:06:25.056866: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/rzangef/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
2026-10-08 19:06:32.378466: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find

Python: 3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]
Python executable: /usr/bin/python3
TensorFlow: 2.20.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


2026-10-08 19:06:37.268628: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [1]:
import sys
import tensorflow as tf

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

2026-10-08 19:08:55.903026: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-08 19:08:56.587985: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-08 19:08:56.588324: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-08 19:08:56.728928: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-08 19:08:57.018996: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-08 19:08:57.023418: I tensorflow/core/platform/cpu_feature_guard.cc:1

Python: 3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]
Python executable: /home/rzangef/SSD-PPE/.venv/bin/python
TensorFlow: 2.15.1
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [2]:
import sys

print("Python executable:")
print(sys.executable)

Python executable:
/home/rzangef/SSD-PPE/.venv/bin/python


In [3]:
from pathlib import Path

SOURCE_DIR = Path.home() / "spark-data/CHVG-Dataset-original/CHVG-Dataset"

DEST_DIR = Path.home() / "SSD-PPE/dataset/CHVG-Dataset-SSD"

print("Dataset original:")
print(SOURCE_DIR)

print("\nDataset SSD:")
print(DEST_DIR)

Dataset original:
/home/rzangef/spark-data/CHVG-Dataset-original/CHVG-Dataset

Dataset SSD:
/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD


In [4]:
images = sorted(SOURCE_DIR.glob("*.jpg"))
annotations = sorted(SOURCE_DIR.glob("*.xml"))

print("Jumlah JPG :", len(images))
print("Jumlah XML :", len(annotations))

Jumlah JPG : 1699
Jumlah XML : 1699


In [5]:
missing_xml = [
    image.name
    for image in images
    if not image.with_suffix(".xml").exists()
]

print("JPG tanpa XML:", len(missing_xml))

if missing_xml:
    print(missing_xml[:10])

JPG tanpa XML: 0


In [6]:
missing_image = [
    xml.name
    for xml in annotations
    if not xml.with_suffix(".jpg").exists()
]

print("XML tanpa JPG:", len(missing_image))

if missing_image:
    print(missing_image[:10])

XML tanpa JPG: 0


In [7]:
for split in ["train", "val", "test"]:
    (DEST_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (DEST_DIR / split / "annotations").mkdir(parents=True, exist_ok=True)

print("Folder dataset SSD siap.")

Folder dataset SSD siap.


In [8]:
import random

TRAIN_COUNT = 1189
VAL_COUNT = 170
TEST_COUNT = 340

SEED = 42

pairs = [
    (image, image.with_suffix(".xml"))
    for image in images
    if image.with_suffix(".xml").exists()
]

print("Jumlah pasangan JPG/XML:", len(pairs))

random.seed(SEED)
random.shuffle(pairs)

train_pairs = pairs[:TRAIN_COUNT]

val_start = TRAIN_COUNT
val_end = TRAIN_COUNT + VAL_COUNT
val_pairs = pairs[val_start:val_end]

test_pairs = pairs[val_end:]

print("Train:", len(train_pairs))
print("Val  :", len(val_pairs))
print("Test :", len(test_pairs))

Jumlah pasangan JPG/XML: 1699
Train: 1189
Val  : 170
Test : 340


In [9]:
import shutil

splits = {
    "train": train_pairs,
    "val": val_pairs,
    "test": test_pairs,
}

for split_name, split_pairs in splits.items():

    image_dest = DEST_DIR / split_name / "images"
    annotation_dest = DEST_DIR / split_name / "annotations"

    for image_path, xml_path in split_pairs:

        shutil.copy2(
            image_path,
            image_dest / image_path.name
        )

        shutil.copy2(
            xml_path,
            annotation_dest / xml_path.name
        )

    print(f"{split_name}: {len(split_pairs)} pasangan disalin")

train: 1189 pasangan disalin
val: 170 pasangan disalin
test: 340 pasangan disalin


In [10]:
for split in ["train", "val", "test"]:

    image_count = len(list((DEST_DIR / split / "images").glob("*.jpg")))
    xml_count = len(list((DEST_DIR / split / "annotations").glob("*.xml")))

    print(
        f"{split:5s} | "
        f"JPG: {image_count:4d} | "
        f"XML: {xml_count:4d}"
    )

train | JPG: 1189 | XML: 1189
val   | JPG:  170 | XML:  170
test  | JPG:  340 | XML:  340


In [11]:
# label_map.pbtxt
from pathlib import Path

label_map_path = DEST_DIR / "label_map.pbtxt"

labels = {
    1: "person",
    2: "vest",
    3: "yellow",
    4: "white",
    5: "glass",
    6: "blue",
    7: "red",
    8: "head",
}

with open(label_map_path, "w") as f:
    for class_id, class_name in labels.items():
        f.write(
            "item {\n"
            f"    id: {class_id}\n"
            f"    name: '{class_name}'\n"
            "}\n\n"
        )

print(f"Label map dibuat: {label_map_path}")

Label map dibuat: /home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt


In [12]:
print(label_map_path.read_text())

item {
    id: 1
    name: 'person'
}

item {
    id: 2
    name: 'vest'
}

item {
    id: 3
    name: 'yellow'
}

item {
    id: 4
    name: 'white'
}

item {
    id: 5
    name: 'glass'
}

item {
    id: 6
    name: 'blue'
}

item {
    id: 7
    name: 'red'
}

item {
    id: 8
    name: 'head'
}




In [13]:
from object_detection.utils import label_map_util

print("Object Detection API: OK")

ModuleNotFoundError: No module named 'object_detection'

In [14]:
from pathlib import Path

models_dir = Path.home() / "SSD-PPE/models"

print("models directory:")
print(models_dir)

print("\nIsi models:")
print([p.name for p in models_dir.iterdir()])

print("\nFolder object_detection yang ditemukan:")
matches = list(models_dir.rglob("object_detection"))
for p in matches[:20]:
    print(p)

models directory:
/home/rzangef/SSD-PPE/models

Isi models:
['.github', 'official', 'CONTRIBUTING.md', 'ISSUES.md', 'README.md', '.git', 'docs', '.gitignore', 'tensorflow_models', 'orbit', 'LICENSE', 'CODEOWNERS', 'AUTHORS']

Folder object_detection yang ditemukan:
/home/rzangef/SSD-PPE/models/official/vision/utils/object_detection


In [15]:
import tensorflow as tf
import sys

print("Python :", sys.version)
print("TensorFlow :", tf.__version__)
print("Python executable :", sys.executable)

Python : 3.10.12 (main, Aug 31 2026, 10:18:17) [GCC 11.4.0]
TensorFlow : 2.15.1
Python executable : /home/rzangef/SSD-PPE/.venv/bin/python


In [16]:
import os
import io
import xml.etree.ElementTree as ET

import tensorflow as tf
from PIL import Image

from object_detection.utils import dataset_util

print("TensorFlow:", tf.__version__)
print("TFRecord utilities: OK")

ModuleNotFoundError: No module named 'object_detection'

In [17]:
import sys

sys.path.insert(0, "/home/rzangef/SSD-PPE/object-detection-api/research")

from object_detection.utils import dataset_util

print("Object Detection API: OK")

Object Detection API: OK


In [18]:
CLASS_MAP = {
    "person": 1,
    "vest": 2,
    "yellow": 3,
    "white": 4,
    "glass": 5,
    "blue": 6,
    "red": 7,
    "head": 8,
}

def create_tf_example(xml_file, image_dir):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    filename = root.find("filename").text
    image_path = os.path.join(image_dir, filename)

    with tf.io.gfile.GFile(image_path, "rb") as fid:
        encoded_image = fid.read()

    image = Image.open(io.BytesIO(encoded_image))
    width, height = image.size

    xmins = []
    xmaxs = []
    ymins = []
    ymaxs = []
    classes_text = []
    classes = []

    for member in root.findall("object"):
        class_name = member.find("name").text

        if class_name not in CLASS_MAP:
            continue

        xmin = float(member.find("bndbox/xmin").text)
        xmax = float(member.find("bndbox/xmax").text)
        ymin = float(member.find("bndbox/ymin").text)
        ymax = float(member.find("bndbox/ymax").text)

        xmins.append(xmin / width)
        xmaxs.append(xmax / width)
        ymins.append(ymin / height)
        ymaxs.append(ymax / height)

        classes_text.append(class_name.encode("utf8"))
        classes.append(CLASS_MAP[class_name])

    tf_example = tf.train.Example(
        features=tf.train.Features(
            feature={
                "image/height": dataset_util.int64_feature(height),
                "image/width": dataset_util.int64_feature(width),
                "image/filename": dataset_util.bytes_feature(
                    filename.encode("utf8")
                ),
                "image/source_id": dataset_util.bytes_feature(
                    filename.encode("utf8")
                ),
                "image/encoded": dataset_util.bytes_feature(encoded_image),
                "image/format": dataset_util.bytes_feature(b"jpg"),
                "image/object/bbox/xmin": dataset_util.float_list_feature(xmins),
                "image/object/bbox/xmax": dataset_util.float_list_feature(xmaxs),
                "image/object/bbox/ymin": dataset_util.float_list_feature(ymins),
                "image/object/bbox/ymax": dataset_util.float_list_feature(ymaxs),
                "image/object/class/text": dataset_util.bytes_list_feature(
                    classes_text
                ),
                "image/object/class/label": dataset_util.int64_list_feature(
                    classes
                ),
            }
        )
    )

    return tf_example

print("Fungsi create_tf_example: OK")

Fungsi create_tf_example: OK


In [19]:
# XML -> TFRecord
def generate_tfrecord(annotation_dir, image_dir, output_path):
    writer = tf.io.TFRecordWriter(output_path)

    xml_files = sorted(
        os.path.join(annotation_dir, f)
        for f in os.listdir(annotation_dir)
        if f.endswith(".xml")
    )

    print(f"Jumlah XML: {len(xml_files)}")

    for i, xml_file in enumerate(xml_files, start=1):
        tf_example = create_tf_example(xml_file, image_dir)
        writer.write(tf_example.SerializeToString())

        if i % 100 == 0:
            print(f"Progress: {i}/{len(xml_files)}")

    writer.close()

    print(f"Selesai: {output_path}")

In [20]:
# Path Dataset
DATASET_DIR = "/home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD"
TFRECORD_DIR = "/home/rzangef/SSD-PPE/dataset/tfrecord"

TRAIN_IMAGES = os.path.join(DATASET_DIR, "train", "images")
TRAIN_ANNOTATIONS = os.path.join(DATASET_DIR, "train", "annotations")
TRAIN_RECORD = os.path.join(TFRECORD_DIR, "train.record")

print("Train images:", TRAIN_IMAGES)
print("Train annotations:", TRAIN_ANNOTATIONS)
print("Output:", TRAIN_RECORD)

Train images: /home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/train/images
Train annotations: /home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/train/annotations
Output: /home/rzangef/SSD-PPE/dataset/tfrecord/train.record


In [21]:
generate_tfrecord(
    annotation_dir=TRAIN_ANNOTATIONS,
    image_dir=TRAIN_IMAGES,
    output_path=TRAIN_RECORD
)

Jumlah XML: 1189
Progress: 100/1189
Progress: 200/1189
Progress: 300/1189
Progress: 400/1189
Progress: 500/1189
Progress: 600/1189
Progress: 700/1189
Progress: 800/1189
Progress: 900/1189
Progress: 1000/1189
Progress: 1100/1189
Selesai: /home/rzangef/SSD-PPE/dataset/tfrecord/train.record


In [22]:
# val.record
VAL_IMAGES = os.path.join(DATASET_DIR, "val", "images")
VAL_ANNOTATIONS = os.path.join(DATASET_DIR, "val", "annotations")
VAL_RECORD = os.path.join(TFRECORD_DIR, "val.record")

generate_tfrecord(
    annotation_dir=VAL_ANNOTATIONS,
    image_dir=VAL_IMAGES,
    output_path=VAL_RECORD
)

Jumlah XML: 170
Progress: 100/170
Selesai: /home/rzangef/SSD-PPE/dataset/tfrecord/val.record


In [23]:
# test.record
TEST_IMAGES = os.path.join(DATASET_DIR, "test", "images")
TEST_ANNOTATIONS = os.path.join(DATASET_DIR, "test", "annotations")
TEST_RECORD = os.path.join(TFRECORD_DIR, "test.record")

generate_tfrecord(
    annotation_dir=TEST_ANNOTATIONS,
    image_dir=TEST_IMAGES,
    output_path=TEST_RECORD
)

Jumlah XML: 340
Progress: 100/340
Progress: 200/340
Progress: 300/340
Selesai: /home/rzangef/SSD-PPE/dataset/tfrecord/test.record


In [24]:
for split in ["train", "val", "test"]:
    record_path = os.path.join(TFRECORD_DIR, f"{split}.record")
    count = sum(1 for _ in tf.data.TFRecordDataset(record_path))
    print(f"{split}: {count} records")

train: 1189 records
val: 170 records
test: 340 records


In [25]:
MODEL_DIR = "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model directory:", MODEL_DIR)

Model directory: /home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320


In [26]:
# Install SSD MobileNet V2 FPNLite 320 x 320
import os
import urllib.request
import tarfile

MODEL_URL = (
    "https://download.tensorflow.org/models/object_detection/tf2/"
    "20200711/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz"
)

MODEL_ARCHIVE = "/home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320.tar.gz"

urllib.request.urlretrieve(MODEL_URL, MODEL_ARCHIVE)

print("Download selesai:", MODEL_ARCHIVE)

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'download.tensorflow.org'. (_ssl.c:1007)>

In [27]:
# Smoke Test
import sys
import subprocess

RESEARCH_DIR = "/home/rzangef/SSD-PPE/object-detection-api/research"
PIPELINE_CONFIG = "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
MODEL_DIR = "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320"
TRAIN_SCRIPT = f"{RESEARCH_DIR}/object_detection/model_main_tf2.py"

result = subprocess.run(
    [
        sys.executable,
        TRAIN_SCRIPT,
        f"--pipeline_config_path={PIPELINE_CONFIG}",
        f"--model_dir={MODEL_DIR}",
        "--num_train_steps=100",
        "--alsologtostderr",
    ],
    cwd=RESEARCH_DIR,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)


2026-10-08 23:38:13.177088: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-08 23:38:13.338659: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-08 23:38:13.338829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-08 23:38:13.354077: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-08 23:38:13.423358: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-08 23:38:13.424460: I tensorflow/core/platform/cpu_feature_guard.cc:

In [28]:
import os
import sys
import subprocess

RESEARCH_DIR = "/home/rzangef/SSD-PPE/object-detection-api/research"
PIPELINE_CONFIG = "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
MODEL_DIR = "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320"
TRAIN_SCRIPT = f"{RESEARCH_DIR}/object_detection/model_main_tf2.py"

env = os.environ.copy()
env["PYTHONPATH"] = RESEARCH_DIR + ":" + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable,
        TRAIN_SCRIPT,
        f"--pipeline_config_path={PIPELINE_CONFIG}",
        f"--model_dir={MODEL_DIR}",
        "--num_train_steps=100",
        "--alsologtostderr",
    ],
    cwd=RESEARCH_DIR,
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)


2026-10-08 23:39:08.976813: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-08 23:39:09.074732: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-08 23:39:09.074850: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-08 23:39:09.078100: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-08 23:39:09.094065: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-08 23:39:09.094460: I tensorflow/core/platform/cpu_feature_guard.cc:

In [29]:
import tf_slim
import inspect
from tf_slim.data import tfexample_decoder

print(inspect.getfile(tfexample_decoder))

/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py


In [30]:
import tf_slim
import os

tf_slim_dir = os.path.dirname(tf_slim.__file__)

!grep -R "control_flow_ops.case" "$tf_slim_dir"

/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py:    image = control_flow_ops.case(
/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder_test.py:          image = control_flow_ops.case(


In [31]:
file = "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py"

with open(file, "r") as f:
    lines = f.readlines()

for i in range(285, 310):
    print(f"{i+1}: {lines[i].rstrip()}")

286: 
287:   def __init__(self,
288:                indices_key=None,
289:                values_key=None,
290:                shape_key=None,
291:                shape=None,
292:                densify=False,
293:                default_value=0):
294:     """Initializes the Tensor handler.
295: 
296:     Args:
297:       indices_key: the name of the TF-Example feature that contains the ids.
298:         Defaults to 'indices'.
299:       values_key: the name of the TF-Example feature that contains the values.
300:         Defaults to 'values'.
301:       shape_key: the name of the TF-Example feature that contains the shape.
302:         If provided it would be used.
303:       shape: the output shape of the SparseTensor. If `shape_key` is not
304:         provided this `shape` would be used.
305:       densify: whether to convert the SparseTensor into a dense Tensor.
306:       default_value: Scalar value to set when making dense for indices not
307:         specified in the `SparseTen

In [32]:
file = "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py"

with open(file, "r") as f:
    for i, line in enumerate(f, 1):
        if "control_flow_ops.case" in line:
            print(f"Baris {i}: {line.strip()}")

Baris 453: image = control_flow_ops.case(


In [33]:
file = "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py"

with open(file, "r") as f:
    lines = f.readlines()

for i in range(440, 475):
    print(f"{i+1}: {lines[i].rstrip()}")

441:           decode_jpeg,
442:           decode_image,
443:           name='cond_jpeg')
444: 
445:     def decode_raw():
446:       """Decodes a raw image."""
447:       return parsing_ops.decode_raw(image_buffer, out_type=self._dtype)
448: 
449:     pred_fn_pairs = [(math_ops.logical_or(
450:         math_ops.equal(image_format, 'raw'),
451:         math_ops.equal(image_format, 'RAW')), decode_raw)]
452: 
453:     image = control_flow_ops.case(
454:         pred_fn_pairs, default=check_jpeg, exclusive=True)
455: 
456:     image.set_shape([None, None, self._channels])
457:     if self._shape is not None:
458:       image = array_ops.reshape(image, self._shape)
459: 
460:     return image
461: 
462: 
463: class BoundingBoxSequence(ItemHandler):
464:   """An ItemHandler that concatenates SparseTensors to Bounding Boxes.
465:   """
466: 
467:   def __init__(self, keys=None, prefix=None, return_dense=True,
468:                default_value=-1.0):
469:     """Initialize the bounding box h

In [34]:
file = "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py"

with open(file, "r") as f:
    for i, line in enumerate(f, 1):
        if i <= 60:
            print(f"{i}: {line.rstrip()}")

1: # coding=utf-8
2: # Copyright 2016 The TensorFlow Authors. All Rights Reserved.
3: #
4: # Licensed under the Apache License, Version 2.0 (the "License");
5: # you may not use this file except in compliance with the License.
6: # You may obtain a copy of the License at
7: #
8: #     http://www.apache.org/licenses/LICENSE-2.0
9: #
10: # Unless required by applicable law or agreed to in writing, software
11: # distributed under the License is distributed on an "AS IS" BASIS,
12: # WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
13: # See the License for the specific language governing permissions and
14: # limitations under the License.
15: # ==============================================================================
16: """Contains the TFExampleDecoder its associated helper classes.
17: 
18: The TFExampleDecode is a DataDecoder used to decode TensorFlow Example protos.
19: In order to do so each requested item must be paired with one or more Example
20: fea

In [35]:
from tensorflow.python.ops import control_flow_ops

print("has case:", hasattr(control_flow_ops, "case"))
print("has cond:", hasattr(control_flow_ops, "cond"))

has case: False
has cond: False


In [36]:
import shutil

file = "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py"
backup = file + ".bak"

shutil.copy2(file, backup)

print("Backup dibuat:")
print(backup)

Backup dibuat:
/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py.bak


In [37]:
from pathlib import Path

file = Path("/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py")

text = file.read_text()

# Tambahkan import TensorFlow
if "import tensorflow as tf" not in text:
    text = text.replace(
        "from tensorflow.python.framework import constant_op",
        "import tensorflow as tf\n\nfrom tensorflow.python.framework import constant_op",
        1
    )

# Ganti API yang tidak tersedia pada TensorFlow 2.15.1
text = text.replace(
    "image = control_flow_ops.case(",
    "image = tf.case(",
    1
)

file.write_text(text)

print("Patch berhasil diterapkan.")
print("Import tensorflow as tf:", "import tensorflow as tf" in text)
print("Menggunakan tf.case:", "image = tf.case(" in text)

Patch berhasil diterapkan.
Import tensorflow as tf: True
Menggunakan tf.case: True


In [38]:
import os
import sys
import subprocess

RESEARCH_DIR = "/home/rzangef/SSD-PPE/object-detection-api/research"
PIPELINE_CONFIG = "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
MODEL_DIR = "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320"
TRAIN_SCRIPT = f"{RESEARCH_DIR}/object_detection/model_main_tf2.py"

env = os.environ.copy()
env["PYTHONPATH"] = RESEARCH_DIR + ":" + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable,
        TRAIN_SCRIPT,
        f"--pipeline_config_path={PIPELINE_CONFIG}",
        f"--model_dir={MODEL_DIR}",
        "--num_train_steps=100",
        "--alsologtostderr",
    ],
    cwd=RESEARCH_DIR,
    env=env,
    text=True,
    capture_output=True
)

print("RETURN CODE:", result.returncode)
print("\n--- OUTPUT TERAKHIR ---")
print((result.stdout + "\n" + result.stderr)[-5000:])

RETURN CODE: 1

--- OUTPUT TERAKHIR ---
*nested_args)
  File "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/python/autograph/impl/api.py", line 693, in wrapper
    raise e.ag_error_metadata.to_exception(e)
  File "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/python/autograph/impl/api.py", line 690, in wrapper
    return converted_call(f, args, kwargs, options=options)
  File "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/python/autograph/impl/api.py", line 439, in converted_call
    result = converted_f(*effective_args, **kwargs)
  File "/tmp/__autograph_generated_file6bryh5vx.py", line 74, in tf__decode
    tensors = ag__.converted_call(ag__.ld(decoder).decode, (ag__.ld(serialized_example),), dict(items=ag__.ld(keys)), fscope)
  File "/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/python/autograph/impl/api.py", line 439, in converted_call
    result = converted_f(*effective_args, **kwargs)
  File "

In [39]:
from pathlib import Path

file = Path("/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py")

for i, line in enumerate(file.read_text().splitlines(), 1):
    if "control_flow_ops." in line:
        print(f"{i}: {line.strip()}")

280: return control_flow_ops.cond(
441: return control_flow_ops.cond(


In [40]:
from pathlib import Path

file = Path("/home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tf_slim/data/tfexample_decoder.py")

text = file.read_text()

text = text.replace(
    "return control_flow_ops.cond(",
    "return tf.cond("
)

file.write_text(text)

print("Patch cond berhasil diterapkan.")
print("control_flow_ops.cond tersisa:", text.count("control_flow_ops.cond"))
print("tf.cond:", text.count("tf.cond"))
print("tf.case:", text.count("tf.case"))

Patch cond berhasil diterapkan.
control_flow_ops.cond tersisa: 0
tf.cond: 2
tf.case: 1


In [41]:
import sys

RESEARCH_DIR = "/home/rzangef/SSD-PPE/object-detection-api/research"

if RESEARCH_DIR not in sys.path:
    sys.path.insert(0, RESEARCH_DIR)

from tf_slim.data import tfexample_decoder

print("tfexample_decoder: OK")

tfexample_decoder: OK


In [42]:
# Smoke Test
import os
import sys
import subprocess

RESEARCH_DIR = "/home/rzangef/SSD-PPE/object-detection-api/research"
PIPELINE_CONFIG = "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
MODEL_DIR = "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320"
TRAIN_SCRIPT = f"{RESEARCH_DIR}/object_detection/model_main_tf2.py"

env = os.environ.copy()
env["PYTHONPATH"] = RESEARCH_DIR + ":" + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable,
        TRAIN_SCRIPT,
        f"--pipeline_config_path={PIPELINE_CONFIG}",
        f"--model_dir={MODEL_DIR}",
        "--num_train_steps=100",
        "--alsologtostderr",
    ],
    cwd=RESEARCH_DIR,
    env=env,
    text=True,
    capture_output=True
)

print("RETURN CODE:", result.returncode)
print("\n--- OUTPUT TERAKHIR ---")
print((result.stdout + "\n" + result.stderr)[-5000:])

RETURN CODE: 0

--- OUTPUT TERAKHIR ---
/SSD-PPE/object-detection-api/research/object_detection/builders/dataset_builder.py:100: parallel_interleave (from tensorflow.python.data.experimental.ops.interleave_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use `tf.data.Dataset.interleave(map_func, cycle_length, block_length, num_parallel_calls=tf.data.AUTOTUNE)` instead. If sloppy execution is desired, use `tf.data.Options.deterministic`.
Instructions for updating:
Use `tf.data.Dataset.map()
W1008 23:52:10.529766 132710532190208 deprecation.py:50] From /home/rzangef/SSD-PPE/object-detection-api/research/object_detection/builders/dataset_builder.py:235: DatasetV1.map_with_legacy_function (from tensorflow.python.data.ops.dataset_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use `tf.data.Dataset.map()
Instructions for updating:
Create a `tf.sparse.SparseTensor` and use `tf.sparse.to_dense` instead.
W1008 23:52:19

In [43]:
from pathlib import Path

MODEL_DIR = Path("/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320")

print("Isi model directory:")
for item in sorted(MODEL_DIR.iterdir()):
    print(f"- {item.name}")

print("\nCheckpoint files:")
for item in sorted(MODEL_DIR.glob("ckpt-*")):
    print(f"- {item.name} ({item.stat().st_size / (1024**2):.2f} MB)")

Isi model directory:
- checkpoint
- ckpt-1.data-00000-of-00001
- ckpt-1.index
- train

Checkpoint files:
- ckpt-1.data-00000-of-00001 (10.01 MB)
- ckpt-1.index (0.03 MB)


In [44]:
from pathlib import Path

checkpoint_file = Path(
    "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320/checkpoint"
)

print(checkpoint_file.read_text())

model_checkpoint_path: "ckpt-1"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_timestamps: 1791478372.2630458
last_preserved_timestamp: 1791478369.735931



In [45]:
# Folder training
from pathlib import Path

MAIN_MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

MAIN_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Folder training utama:")
print(MAIN_MODEL_DIR)

Folder training utama:
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch


In [46]:
from pathlib import Path

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

items = list(MODEL_DIR.iterdir())

print("Jumlah file/folder:", len(items))

for item in items:
    print("-", item.name)

Jumlah file/folder: 0


In [47]:
# training
import os
import sys
import subprocess

RESEARCH_DIR = "/home/rzangef/SSD-PPE/object-detection-api/research"
PIPELINE_CONFIG = "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
MODEL_DIR = "/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch"
TRAIN_SCRIPT = f"{RESEARCH_DIR}/object_detection/model_main_tf2.py"

NUM_TRAIN_STEPS = 23780

env = os.environ.copy()
env["PYTHONPATH"] = RESEARCH_DIR + ":" + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable,
        TRAIN_SCRIPT,
        f"--pipeline_config_path={PIPELINE_CONFIG}",
        f"--model_dir={MODEL_DIR}",
        f"--num_train_steps={NUM_TRAIN_STEPS}",
        "--alsologtostderr",
    ],
    cwd=RESEARCH_DIR,
    env=env,
    text=True
)

print("\nRETURN CODE:", result.returncode)

2026-10-09 00:00:44.985503: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-09 00:00:45.086349: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-09 00:00:45.086482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-09 00:00:45.090718: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-09 00:00:45.110858: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-09 00:00:45.111506: I tensorflow/core/platform/cpu_feature_guard.cc:1


RETURN CODE: 0


In [48]:
from pathlib import Path

folders = {
    "Smoke test": Path(
        "/home/rzangef/SSD-PPE/training/"
        "ssd_mobilenet_v2_fpnlite_320x320"
    ),
    "Training utama": Path(
        "/home/rzangef/SSD-PPE/training/"
        "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
    ),
}

for name, folder in folders.items():
    print(f"\n{name}:")
    checkpoint = folder / "checkpoint"

    if checkpoint.exists():
        print(checkpoint.read_text())
    else:
        print("File checkpoint belum ditemukan")


Smoke test:
model_checkpoint_path: "ckpt-1"
all_model_checkpoint_paths: "ckpt-1"
all_model_checkpoint_timestamps: 1791478372.2630458
last_preserved_timestamp: 1791478369.735931


Training utama:
model_checkpoint_path: "ckpt-24"
all_model_checkpoint_paths: "ckpt-18"
all_model_checkpoint_paths: "ckpt-19"
all_model_checkpoint_paths: "ckpt-20"
all_model_checkpoint_paths: "ckpt-21"
all_model_checkpoint_paths: "ckpt-22"
all_model_checkpoint_paths: "ckpt-23"
all_model_checkpoint_paths: "ckpt-24"
all_model_checkpoint_timestamps: 1791483855.7287722
all_model_checkpoint_timestamps: 1791484142.371906
all_model_checkpoint_timestamps: 1791484431.495789
all_model_checkpoint_timestamps: 1791484718.3139415
all_model_checkpoint_timestamps: 1791485007.5112097
all_model_checkpoint_timestamps: 1791485294.1478777
all_model_checkpoint_timestamps: 1791485586.6523128
last_preserved_timestamp: 1791478895.114577



In [49]:
from pathlib import Path

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

for suffix in [".index", ".data-00000-of-00001"]:
    file = MODEL_DIR / f"ckpt-24{suffix}"
    print(
        f"{file.name}:",
        "ADA" if file.exists() else "TIDAK DITEMUKAN",
        f"({file.stat().st_size / (1024**2):.2f} MB)"
        if file.exists() else ""
    )

ckpt-24.index: ADA (0.05 MB)
ckpt-24.data-00000-of-00001: ADA (19.85 MB)


In [50]:
from pathlib import Path
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

CONFIG_PATH = Path(
    "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
)

config = pipeline_pb2.TrainEvalPipelineConfig()
with CONFIG_PATH.open() as f:
    text_format.Merge(f.read(), config)

print("Jumlah kelas:", config.model.ssd.num_classes)
print("Batch evaluasi:", config.eval_config.batch_size)
print("Checkpoint awal dari config:", config.train_config.fine_tune_checkpoint)

print("\nInput evaluasi:")
for reader in config.eval_input_reader:
    print("Label map:", reader.label_map_path)
    for item in reader.tf_record_input_reader.input_path:
        print("TFRecord:", item)

print("\nCheckpoint training utama:")
print(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-24"
)

Jumlah kelas: 8
Batch evaluasi: 1
Checkpoint awal dari config: /home/rzangef/SSD-PPE/models/ssd_mobilenet_v2_fpnlite_320x320/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0

Input evaluasi:
Label map: /home/rzangef/SSD-PPE/dataset/CHVG-Dataset-SSD/label_map.pbtxt
TFRecord: /home/rzangef/SSD-PPE/dataset/tfrecord/val.record

Checkpoint training utama:
/home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-24


In [51]:
import os
import sys
import subprocess

RESEARCH_DIR = "/home/rzangef/SSD-PPE/object-detection-api/research"
PIPELINE_CONFIG = "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
MODEL_DIR = (
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)
EVAL_DIR = MODEL_DIR + "/eval"

os.makedirs(EVAL_DIR, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = RESEARCH_DIR + ":" + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable,
        f"{RESEARCH_DIR}/object_detection/model_main_tf2.py",
        f"--pipeline_config_path={PIPELINE_CONFIG}",
        f"--model_dir={MODEL_DIR}",
        f"--checkpoint_dir={MODEL_DIR}",
        "--alsologtostderr",
    ],
    cwd=RESEARCH_DIR,
    env=env,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("\n--- OUTPUT TERAKHIR ---")
print((result.stdout + "\n" + result.stderr)[-6000:])

RETURN CODE: 0

--- OUTPUT TERAKHIR ---
W1009 07:07:34.803667 133712068124672 deprecation.py:50] From /home/rzangef/SSD-PPE/.venv/lib/python3.10/site-packages/tensorflow/python/util/dispatch.py:1260: to_int64 (from tensorflow.python.ops.math_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use `tf.cast` instead.
INFO:tensorflow:Finished eval step 0
I1009 07:07:34.834235 133712068124672 model_lib_v2.py:966] Finished eval step 0
Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf

In [1]:
# File hasil evaluasi
from pathlib import Path

EVAL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch/eval"
)

print("Folder evaluasi:", EVAL_DIR)
print("Ada:", EVAL_DIR.exists())

if EVAL_DIR.exists():
    for item in sorted(EVAL_DIR.rglob("*")):
        if item.is_file():
            print(item.relative_to(EVAL_DIR))

Folder evaluasi: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/eval
Ada: True
events.out.tfevents.1791504421.LAPTOP-ODRK022O.6724.0.v2


In [2]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

EVENT_FILE = (
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch/eval/"
    "events.out.tfevents.1791504421.LAPTOP-ODRK022O.6724.0.v2"
)

ea = EventAccumulator(EVENT_FILE)
ea.Reload()

print("Tag evaluasi yang tersimpan:")
for tag in ea.Tags()["scalars"]:
    print("-", tag)

print("\nNilai metrik terakhir:")
for tag in ea.Tags()["scalars"]:
    events = ea.Scalars(tag)
    if events:
        last = events[-1]
        print(f"{tag}: {last.value:.6f} (step {last.step})")

2026-10-09 22:35:50.248665: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-09 22:35:51.018846: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-09 22:35:51.019217: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-09 22:35:51.167686: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-09 22:35:51.480858: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-09 22:35:51.484744: I tensorflow/core/platform/cpu_feature_guard.cc:1

Tag evaluasi yang tersimpan:

Nilai metrik terakhir:


In [3]:
from pathlib import Path

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

print("=== CHECKPOINT ===")
checkpoint_file = MODEL_DIR / "checkpoint"

if checkpoint_file.exists():
    print(checkpoint_file.read_text())
else:
    print("File checkpoint tidak ditemukan.")

print("\n=== FILE EVALUASI ===")
eval_dir = MODEL_DIR / "eval"

for item in sorted(eval_dir.glob("*")):
    if item.is_file():
        print(item.name, "|", item.stat().st_size, "bytes")

=== CHECKPOINT ===
model_checkpoint_path: "ckpt-24"
all_model_checkpoint_paths: "ckpt-18"
all_model_checkpoint_paths: "ckpt-19"
all_model_checkpoint_paths: "ckpt-20"
all_model_checkpoint_paths: "ckpt-21"
all_model_checkpoint_paths: "ckpt-22"
all_model_checkpoint_paths: "ckpt-23"
all_model_checkpoint_paths: "ckpt-24"
all_model_checkpoint_timestamps: 1791483855.7287722
all_model_checkpoint_timestamps: 1791484142.371906
all_model_checkpoint_timestamps: 1791484431.495789
all_model_checkpoint_timestamps: 1791484718.3139415
all_model_checkpoint_timestamps: 1791485007.5112097
all_model_checkpoint_timestamps: 1791485294.1478777
all_model_checkpoint_timestamps: 1791485586.6523128
last_preserved_timestamp: 1791478895.114577


=== FILE EVALUASI ===
events.out.tfevents.1791504421.LAPTOP-ODRK022O.6724.0.v2 | 2239221 bytes


In [4]:
import subprocess
import sys
from pathlib import Path

RESEARCH_DIR = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)
SCRIPT = RESEARCH_DIR / "object_detection" / "model_main_tf2.py"

result = subprocess.run(
    [sys.executable, str(SCRIPT), "--help"],
    cwd=str(RESEARCH_DIR),
    capture_output=True,
    text=True,
    timeout=60
)

print("RETURN CODE:", result.returncode)

for line in (result.stdout + "\n" + result.stderr).splitlines():
    if any(word in line.lower() for word in [
        "checkpoint", "eval", "wait", "pipeline_config_path"
    ]):
        print(line)

RETURN CODE: 1


In [5]:
import subprocess
import sys
from pathlib import Path

RESEARCH_DIR = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)
SCRIPT = RESEARCH_DIR / "object_detection" / "model_main_tf2.py"

result = subprocess.run(
    [sys.executable, str(SCRIPT), "--help"],
    cwd=str(RESEARCH_DIR),
    capture_output=True,
    text=True,
    timeout=60
)

print("RETURN CODE:", result.returncode)
print("\n=== STDOUT ===")
print(result.stdout[-5000:] or "(kosong)")
print("\n=== STDERR ===")
print(result.stderr[-10000:] or "(kosong)")

RETURN CODE: 1

=== STDOUT ===
(kosong)

=== STDERR ===
2026-10-09 23:19:22.474436: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-09 23:19:22.586838: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-09 23:19:22.586993: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-09 23:19:22.590209: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-09 23:19:22.607440: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-09 23:19:22.6

In [6]:
# Lokasi Object Detection API
import sys
from pathlib import Path

RESEARCH_DIR = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)

print("Research directory ada:", RESEARCH_DIR.is_dir())
print("Folder object_detection ada:",
      (RESEARCH_DIR / "object_detection").is_dir())
print("Python executable:", sys.executable)
print("Research directory di sys.path:",
      str(RESEARCH_DIR) in sys.path)

Research directory ada: True
Folder object_detection ada: True
Python executable: /home/rzangef/SSD-PPE/.venv/bin/python
Research directory di sys.path: False


In [7]:
import os
import subprocess
import sys
from pathlib import Path

RESEARCH_DIR = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)
SCRIPT = RESEARCH_DIR / "object_detection" / "model_main_tf2.py"

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(RESEARCH_DIR)
    + os.pathsep
    + env.get("PYTHONPATH", "")
).rstrip(os.pathsep)

result = subprocess.run(
    [sys.executable, str(SCRIPT), "--help"],
    cwd=str(RESEARCH_DIR),
    env=env,
    capture_output=True,
    text=True,
    timeout=60
)

print("RETURN CODE:", result.returncode)
print("\n=== OUTPUT TERAKHIR ===")
print((result.stdout + "\n" + result.stderr)[-7000:])

RETURN CODE: 1

=== OUTPUT TERAKHIR ===
Creates and runs TF2 object detection models.

For local training/evaluation run:
PIPELINE_CONFIG_PATH=path/to/pipeline.config
MODEL_DIR=/tmp/model_outputs
NUM_TRAIN_STEPS=10000
SAMPLE_1_OF_N_EVAL_EXAMPLES=1
python model_main_tf2.py -- \
  --model_dir=$MODEL_DIR --num_train_steps=$NUM_TRAIN_STEPS \
  --sample_1_of_n_eval_examples=$SAMPLE_1_OF_N_EVAL_EXAMPLES \
  --pipeline_config_path=$PIPELINE_CONFIG_PATH \
  --alsologtostderr

flags:

/home/rzangef/SSD-PPE/object-detection-api/research/object_detection/model_main_tf2.py:
  --checkpoint_dir: Path to directory holding a checkpoint.  If `checkpoint_dir`
    is provided, this binary operates in eval-only mode, writing resulting
    metrics to `model_dir`.
  --checkpoint_every_n: Integer defining how often we checkpoint.
    (default: '1000')
    (an integer)
  --[no]eval_on_train_data: Enable evaluating on train data (only supported in
    distributed training).
    (default: 'false')
  --eval_time

In [8]:
import os
import subprocess
import sys
from pathlib import Path

RESEARCH_DIR = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)
SCRIPT = RESEARCH_DIR / "object_detection" / "model_main_tf2.py"

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)
EVAL_OUTPUT_DIR = MODEL_DIR / "eval_ckpt24"

EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(RESEARCH_DIR) + os.pathsep + env.get("PYTHONPATH", "")
).rstrip(os.pathsep)

cmd = [
    sys.executable,
    str(SCRIPT),
    f"--pipeline_config_path={MODEL_DIR.parent / 'pipeline_ppe.config'}",
    f"--model_dir={EVAL_OUTPUT_DIR}",
    f"--checkpoint_dir={MODEL_DIR}",
    "--eval_timeout=120",
    "--alsologtostderr",
]

print("Perintah evaluasi dimulai...")
print("Checkpoint directory:", MODEL_DIR)
print("Output evaluasi baru:", EVAL_OUTPUT_DIR)

result = subprocess.run(
    cmd,
    cwd=str(RESEARCH_DIR),
    env=env,
    capture_output=True,
    text=True,
    timeout=600
)

print("\nRETURN CODE:", result.returncode)
print("\n=== OUTPUT EVALUASI ===")
print((result.stdout + "\n" + result.stderr)[-12000:])

Perintah evaluasi dimulai...
Checkpoint directory: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch
Output evaluasi baru: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/eval_ckpt24

RETURN CODE: 1

=== OUTPUT EVALUASI ===

2026-10-09 23:22:19.301935: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-09 23:22:19.390041: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-09 23:22:19.390159: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-09 23:22:19.393109: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for p

In [9]:
import os
import subprocess
import sys
from pathlib import Path

RESEARCH_DIR = Path(
    "/home/rzangef/SSD-PPE/object-detection-api/research"
)
SCRIPT = RESEARCH_DIR / "object_detection" / "model_main_tf2.py"

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)
PIPELINE_CONFIG = Path(
    "/home/rzangef/SSD-PPE/models/pipeline_ppe.config"
)
EVAL_OUTPUT_DIR = MODEL_DIR / "eval_ckpt24"

print("Config tersedia:", PIPELINE_CONFIG.is_file())
print("Checkpoint tersedia:", (MODEL_DIR / "checkpoint").is_file())

assert PIPELINE_CONFIG.is_file(), "File pipeline config tidak ditemukan."
assert (MODEL_DIR / "checkpoint").is_file(), "File checkpoint tidak ditemukan."

EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = (
    str(RESEARCH_DIR) + os.pathsep + env.get("PYTHONPATH", "")
).rstrip(os.pathsep)

cmd = [
    sys.executable,
    str(SCRIPT),
    f"--pipeline_config_path={PIPELINE_CONFIG}",
    f"--model_dir={EVAL_OUTPUT_DIR}",
    f"--checkpoint_dir={MODEL_DIR}",
    "--eval_timeout=120",
    "--alsologtostderr",
]

result = subprocess.run(
    cmd,
    cwd=str(RESEARCH_DIR),
    env=env,
    capture_output=True,
    text=True,
    timeout=600
)

print("\nRETURN CODE:", result.returncode)
print("\n=== OUTPUT EVALUASI ===")
print((result.stdout + "\n" + result.stderr)[-12000:])

Config tersedia: True
Checkpoint tersedia: True

RETURN CODE: 0

=== OUTPUT EVALUASI ===
amples: None
I1009 23:23:40.603223 136592169517056 config_util.py:552] Maybe overwriting sample_1_of_n_eval_examples: None
INFO:tensorflow:Maybe overwriting use_bfloat16: False
I1009 23:23:40.603359 136592169517056 config_util.py:552] Maybe overwriting use_bfloat16: False
INFO:tensorflow:Maybe overwriting eval_num_epochs: 1
I1009 23:23:40.603492 136592169517056 config_util.py:552] Maybe overwriting eval_num_epochs: 1
W1009 23:23:40.603683 136592169517056 model_lib_v2.py:1106] Expected number of evaluation epochs is 1, but instead encountered `eval_on_train_input_config.num_epochs` = 0. Overwriting `num_epochs` to 1.
INFO:tensorflow:Reading unweighted datasets: ['/home/rzangef/SSD-PPE/dataset/tfrecord/val.record']
I1009 23:23:40.736449 136592169517056 dataset_builder.py:162] Reading unweighted datasets: ['/home/rzangef/SSD-PPE/dataset/tfrecord/val.record']
INFO:tensorflow:Reading record datasets for

In [10]:
import tensorflow as tf
from pathlib import Path

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

checkpoint_path = str(MODEL_DIR / "ckpt-24")

reader = tf.train.load_checkpoint(checkpoint_path)
variables = reader.get_variable_to_shape_map()

print("Checkpoint:", checkpoint_path)
print("Global step:", reader.get_tensor("global_step"))
print("\nVariabel global_step ditemukan:", "global_step" in variables)

Checkpoint: /home/rzangef/SSD-PPE/training/ssd_mobilenet_v2_fpnlite_320x320_20epoch/ckpt-24


NotFoundError: Key global_step not found in checkpoint

In [11]:
import tensorflow as tf
from pathlib import Path

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

checkpoint_path = str(MODEL_DIR / "ckpt-24")
reader = tf.train.load_checkpoint(checkpoint_path)
variables = reader.get_variable_to_shape_map()

print("Jumlah variabel:", len(variables))
print("\nVariabel yang mengandung kata 'step':")

step_vars = [name for name in variables if "step" in name.lower()]

if step_vars:
    for name in step_vars:
        print(name, "=", reader.get_tensor(name))
else:
    print("Tidak ditemukan.")

print("\nLima belas nama variabel pertama:")
for name in list(variables.keys())[:15]:
    print(name, "| shape:", variables[name])

Jumlah variabel: 622

Variabel yang mengandung kata 'step':
step/.ATTRIBUTES/VARIABLE_VALUE = 23000

Lima belas nama variabel pertama:
step/.ATTRIBUTES/VARIABLE_VALUE | shape: []
optimizer/decay/.ATTRIBUTES/VARIABLE_VALUE | shape: []
model/_feature_extractor/classification_backbone/layer_with_weights-99/moving_variance/.ATTRIBUTES/VARIABLE_VALUE | shape: [960]
model/_feature_extractor/classification_backbone/layer_with_weights-99/gamma/.OPTIMIZER_SLOT/optimizer/momentum/.ATTRIBUTES/VARIABLE_VALUE | shape: [960]
model/_feature_extractor/classification_backbone/layer_with_weights-99/gamma/.ATTRIBUTES/VARIABLE_VALUE | shape: [960]
model/_feature_extractor/classification_backbone/layer_with_weights-99/beta/.OPTIMIZER_SLOT/optimizer/momentum/.ATTRIBUTES/VARIABLE_VALUE | shape: [960]
model/_feature_extractor/classification_backbone/layer_with_weights-97/moving_variance/.ATTRIBUTES/VARIABLE_VALUE | shape: [960]
model/_feature_extractor/classification_backbone/layer_with_weights-97/moving_mean

In [12]:
import tensorflow as tf
from pathlib import Path

MODEL_DIR = Path(
    "/home/rzangef/SSD-PPE/training/"
    "ssd_mobilenet_v2_fpnlite_320x320_20epoch"
)

checkpoint_file = MODEL_DIR / "checkpoint"

for line in checkpoint_file.read_text().splitlines():
    if line.startswith("all_model_checkpoint_paths:"):
        name = line.split('"')[1]
        path = str(MODEL_DIR / name)

        reader = tf.train.load_checkpoint(path)
        variables = reader.get_variable_to_shape_map()

        step_key = "step/.ATTRIBUTES/VARIABLE_VALUE"
        step = (
            reader.get_tensor(step_key)
            if step_key in variables
            else "tidak ditemukan"
        )

        print(f"{name}: global step = {step}")

ckpt-18: global step = 17000
ckpt-19: global step = 18000
ckpt-20: global step = 19000
ckpt-21: global step = 20000
ckpt-22: global step = 21000
ckpt-23: global step = 22000
ckpt-24: global step = 23000
